In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os
import time

load_dotenv()

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY")
)

# ---------------- STATE ----------------

class StreamState(TypedDict):
    topic: str
    joke: str
    explanation: str

# ---------------- NODES ----------------

def generate_joke(state: StreamState):

    print("Generating Joke...")
    time.sleep(2)

    response = llm.invoke(
        f"Generate a funny joke about {state['topic']}"
    ).content

    return {
        "joke": response
    }


def explain_joke(state: StreamState):

    print("Generating Explanation...")
    time.sleep(2)

    response = llm.invoke(
        f"Explain this joke:\n\n{state['joke']}"
    ).content

    return {
        "explanation": response
    }

# ---------------- GRAPH ----------------

graph = StateGraph(StreamState)

graph.add_node("generate_joke", generate_joke)
graph.add_node("explain_joke", explain_joke)

graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke", "explain_joke")
graph.add_edge("explain_joke", END)

workflow = graph.compile()

initial_state = {
    "topic": "Python"
}

## 1️⃣ stream_mode="updates" (Streams only what each node returns.)

In [2]:
for event in workflow.stream(
    initial_state,
    stream_mode="updates"
):
    print("="*60)
    print(event)

Generating Joke...
{'generate_joke': {'joke': 'Why did the Python script go to therapy?\n\nBecause it had a lot of "indent"-ernal issues and was struggling to "wrap" its head around its problems! (get it? indent, like indentation in Python code?)'}}
Generating Explanation...
{'explain_joke': {'explanation': 'A programming joke.\n\nThis joke is a play on words, using Python programming terminology to create a pun. Here\'s how it works:\n\n1. **"Indent"-ernal issues**: In Python, indentation is used to define the structure of the code, such as loops, conditional statements, and functions. The joke replaces "internal" with "indent-ernal" to reference this programming concept. The wordplay implies that the Python script has internal problems, but also references the code\'s indentation.\n2. **"Wrap" its head around its problems**: This phrase is an idiom that means to understand or come to terms with something. However, in Python, the `wrap` function is not directly related to this phrase.

## 2️⃣ stream_mode="values" (Streams the entire state after every node.)

In [3]:
for event in workflow.stream(
    initial_state,
    stream_mode="values"
):
    print("="*60)
    print(event)

{'topic': 'Python'}
Generating Joke...
{'topic': 'Python', 'joke': 'Why did the Python script go to therapy?\n\nBecause it had a lot of "indent"-ational issues and was struggling to "break" out of its loop!'}
Generating Explanation...
{'topic': 'Python', 'joke': 'Why did the Python script go to therapy?\n\nBecause it had a lot of "indent"-ational issues and was struggling to "break" out of its loop!', 'explanation': 'A joke about Python programming.\n\nThis joke is a play on words, using programming terminology to create a pun. Here\'s a breakdown:\n\n* "Indent"-ational issues: In Python, indentation (spaces or tabs) is used to denote block-level structure. However, the word "indent" sounds similar to "emotional" or "psychological" issues, which are often discussed in therapy. So, the joke is saying that the Python script has emotional issues, but using a word that\'s specific to Python programming.\n* "Break" out of its loop: In Python, the `break` statement is used to exit a loop pre

## 3️⃣ stream_mode="messages" (Streams LLM tokens while they're being generated.)

In [5]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY"),
    streaming=True
)


for token, metadata in workflow.stream(
    initial_state,
    stream_mode="messages"
):
    print(token.content, end="")

Generating Joke...
Why did the Python script go to therapy?

Because it had a lot of "indent"-al issues and was struggling to "wrap" its head around its problems! (get it? indent, like the indentation in Python code?)Generating Explanation...
A programming joke.

The joke is a play on words, using Python programming concepts to make a pun. Here's how it works:

* "Indent"-al issues: In Python, indentation is used to define the structure of the code, such as blocks within loops or conditional statements. The joke is saying that the Python script has emotional or psychological issues (like a person would), but replaces "den" (as in, "internal") with "dent", referencing the code indentation.
* "Wrap its head around": This is a common idiomatic expression meaning to understand or come to terms with something. However, in Python, the `wrap` function is also used to, well, wrap text or other data. So, the joke is making a wordplay on this phrase to tie it back to Python.

The humor comes fro

## 4️⃣ stream_mode="debug" (Shows everything happening internally.)

In [6]:
for event in workflow.stream(
    initial_state,
    stream_mode="debug"
):
    print(event)

{'step': 1, 'timestamp': '2026-07-23T08:19:43.619182+00:00', 'type': 'task', 'payload': {'id': '0ecb3aa8-7e84-181c-5959-c7c963386d45', 'name': 'generate_joke', 'input': {'topic': 'Python'}, 'triggers': ('branch:to:generate_joke',)}}
Generating Joke...
{'step': 1, 'timestamp': '2026-07-23T08:19:45.982269+00:00', 'type': 'task_result', 'payload': {'id': '0ecb3aa8-7e84-181c-5959-c7c963386d45', 'name': 'generate_joke', 'error': None, 'result': {'joke': 'Why did the Python go to the doctor?\n\nBecause it had a little "indent"-ational crisis and was feeling a little "snake"-y! (get it? like the indentation in Python code?)'}, 'interrupts': []}}
{'step': 2, 'timestamp': '2026-07-23T08:19:45.982269+00:00', 'type': 'task', 'payload': {'id': '3d95c2ae-9d22-9ac1-0a4a-6af33089aa14', 'name': 'explain_joke', 'input': {'topic': 'Python', 'joke': 'Why did the Python go to the doctor?\n\nBecause it had a little "indent"-ational crisis and was feeling a little "snake"-y! (get it? like the indentation in

## 5️⃣ stream_mode="custom" (This lets you decide what gets streamed.)

In [ ]:
from langgraph.config import get_stream_writer

def generate_joke(state):

    writer = get_stream_writer()

    writer("Generating Joke...")

    response = llm.invoke(
        f"Generate a joke about {state['topic']}"
    ).content

    writer("Joke Generated!")

    return {
        "joke": response    
    }

for event in workflow.stream(
    initial_state,
    stream_mode="custom"
):
    print(event)

Generating Joke...
Generating Explanation...
